In [22]:
import torch
import torch.nn as nn
import torch.nn.functional as F

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.13.0+cu126
CUDA available: True


In [23]:
class SimpleCNN(nn.Module):

    def __init__(self, num_classes=10):
        super().__init__()

        self.conv1 = nn.Conv2d(
            3, 32,
            kernel_size=3,
            padding=1
        )

        self.conv2 = nn.Conv2d(
            32, 64,
            kernel_size=3,
            padding=1
        )

        self.pool = nn.MaxPool2d(2)

        self.fc = nn.Linear(
            64 * 8 * 8,
            num_classes
        )

    def forward(self, x):

        x = F.relu(self.conv1(x))
        x = self.pool(x)

        x = F.relu(self.conv2(x))
        x = self.pool(x)

        x = torch.flatten(x, 1)

        x = self.fc(x)

        return x

In [24]:
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP


def setup(rank, world_size):

    dist.init_process_group(
        backend="nccl",
        rank=rank,
        world_size=world_size
    )


def cleanup():

    dist.destroy_process_group()

In [25]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    torch.cuda.set_device(device)

if not dist.is_initialized():
    dist.init_process_group(
        backend="nccl" if torch.cuda.is_available() else "gloo",
        init_method="tcp://127.0.0.1:29501",
        rank=0,
        world_size=1,
    )

model = model.to(device)

model = DDP(
    model,
    device_ids=[device.index] if torch.cuda.is_available() else None,
)

RuntimeError: Distributed package doesn't have NCCL built in

In [ ]:
print(type(model))
print(isinstance(model, DDP))
print(model)